# flair-landcover-transfer — does a French land-cover model work on Ukrainian tiles?

**Status:** `exploring`

`IGNF/FLAIR-INC_rgb_15cl_resnet34-fpn` — FPN + ResNet-34, **RGB only** (Esri
tiles have no NIR and no elevation), trained by IGN on 174,700 photo-interpreted
512x512 patches of French BD ORTHO at **0.2 m**. Licence Etalab 2.0.
Reported per-class IoU: Water 83.1, Building 76.3, Brushwood 29.9, mIoU 59.3.

**This probe gates every aerial-pretrained-model route.** If a model this large
and this well-supervised cannot find buildings over Ukrainian Esri tiles, neither
will LandCover.ai, and a day of training is saved.

It is **self-falsifying**, which probe 1 was not and probe 4 was: `Building` and
`Water` are exactly the two classes where OSM labels are measured good. A model
that misses buildings OSM proves are there cannot be trusted on anything else.

### The two risks, named before running

1. **GSD: model expects 0.2 m, tiles are 0.386 m/px (1.93x).** Both ways of
   feeding it are flawed — upsampling fabricates detail (the exact failure in
   `pocs/drone-frame-boundaries`, where a 131 px crop shown at 512 px returned
   nothing), feeding native halves apparent object size. So run **both**, one
   variable, same tiles.
2. **Sensor: BD ORTHO is aircraft, Esri is a satellite mosaic.** The card says
   outright that generalisation issues can occur at other resolutions and sensors.

### What would make this `failed`

- `Building` or `Water` IoU at or below its null baseline at both GSDs.
- Masks that follow the inference grid rather than the imagery (`bug-016` —
  look at the figure before writing anything).
- Predictions disagreeing across two dates of the *same ground* more than they
  agree — a model keying on season, not structure.

Run from `src/` with the `uavloc` env.
`papermill pocs/flair-landcover-transfer/probe.ipynb pocs/flair-landcover-transfer/runs/probe-$(date +%F).ipynb -p n 9`

In [ ]:
# papermill parameters
SLUG = "flair-landcover-transfer"
n = 9                       # locations
gsd_modes = ["native", "upsampled"]
repo = "IGNF/FLAIR-INC_rgb_15cl_resnet34-fpn"
patch = 512
overlap = 64
flair_mpp = 0.2             # what the model was trained at
cross_date = True           # the no-ground-truth consistency check

## Setup

`segmentation_models_pytorch` and `timm` are **not** in the `uavloc` env. Install
under the existing pin — torch 2.2.2 / torchvision 0.17.2 — or this quietly drags
torch forward and breaks `ultralytics`.

```
/Users/vladyslavtaraban/miniconda3/envs/uavloc/bin/pip install \
    "segmentation-models-pytorch==0.3.4" "timm==0.9.16" \
    -c <(printf 'torch==2.2.2\ntorchvision==0.17.2\n')
```

In [ ]:
import gzip, io, json, sys, time, urllib.request, warnings
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

sys.path.insert(0, str(Path.cwd() / "pocs" / "map-derived-labels"))
import align, osm                              # noqa: E402
from tiles import TileGrid                     # noqa: E402

ROOT = Path("data/derived/ua-village-multidate")
HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"
FIGS.mkdir(parents=True, exist_ok=True)


def locations():
    return json.loads((ROOT / "index.json").read_text())["built"]


def load_location(name, date="current"):
    """(image, grid, meta) at full resolution — same path build.py rasterised through."""
    meta = json.loads((ROOT / name / "meta.json").read_text())
    img = Image.open(ROOT / name / f"{date}.jpg").convert("RGB")
    return img, TileGrid(**meta["grid"]), meta


def osm_mask(name, cls):
    """One class from the dataset's own label.png, shifted by its measured offset.

    The build recorded `label_offset` per location. Charging the dataset's known
    2.84 m label error to the thing under test would be unfair, so undo it here.
    """
    meta = json.loads((ROOT / name / "meta.json").read_text())
    lab = np.array(Image.open(ROOT / name / "label.png"))
    m = lab == osm.CLASSES[cls]
    off = meta.get("label_offset") or {}
    dx, dy = int(off.get("dx_px", 0)), int(off.get("dy_px", 0))
    if dx or dy:
        m = np.roll(np.roll(m, dy, axis=0), dx, axis=1)
    return m


def iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else float("nan")


def null_iou(pred, truth, k=20, seed=0):
    """Same mask, position destroyed. Anything not beating this is an artifact
    of how much ground got painted (bug-016)."""
    rng = np.random.default_rng(seed)
    h, w = pred.shape
    return float(np.median([
        iou(np.roll(np.roll(pred, rng.integers(h // 8, h - h // 8), 0),
                    rng.integers(w // 8, w - w // 8), 1), truth)
        for _ in range(k)
    ]))

In [ ]:
import torch
import segmentation_models_pytorch as smp
from huggingface_hub import hf_hub_download

print("torch", torch.__version__, "| smp", smp.__version__)
torch.set_num_threads(6)        # 6 physical cores on this machine

# FLAIR's nomenclature is **19 classes**, not 15 — the "15cl" in the repo name
# counts the ACTIVE ones. The head emits 19 logits and four were trained with
# weight 0 (README: clear cut, mixed, ligneous, other). Verified against
# `criterion.weight` in the checkpoint, which is [1]*14 + [0,0,0,1,0].
FLAIR_CLASSES = [
    "building", "pervious surface", "impervious surface", "bare soil", "water",
    "coniferous", "deciduous", "brushwood", "vineyard", "herbaceous vegetation",
    "agricultural land", "plowed land", "swimming pool", "snow",
    "clear cut", "mixed", "ligneous",          # deactivated
    "greenhouse",
    "other",                                    # deactivated
]
DEAD = {"clear cut", "mixed", "ligneous", "other"}
DEAD_IDX = [FLAIR_CLASSES.index(c) for c in DEAD]
# how they fold onto our 5-class set
TO_OURS = {
    "building": ["building"],
    "water": ["water", "swimming pool"],
    "woody": ["coniferous", "deciduous", "brushwood"],
    "track": ["impervious surface"],          # FLAIR has no road class — see notes
}
MEAN = np.array([105.08, 110.87, 101.82], np.float32)
STD = np.array([52.17, 45.38, 44.00], np.float32)

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files(repo)
print("repo files:", files)

weights = next((f for f in files if f.endswith((".ckpt", ".pth", ".pt", ".safetensors"))), None)
assert weights, f"no weight file found in {repo}: {files}"
path = hf_hub_download(repo, filename=weights)
print("weights ->", path)

# Keys are prefixed `model.seg_model.` and the checkpoint also carries
# `criterion.weight`, which is not part of the network. Stripping only `model.`
# leaves `seg_model.*`, which matches nothing and silently yields a
# randomly-initialised model — the bug-018 shape of failure.
PREFIX = "model.seg_model."
raw = torch.load(path, map_location="cpu")
state = {k[len(PREFIX):]: v for k, v in raw.items() if k.startswith(PREFIX)}
assert state, f"prefix {PREFIX!r} matched nothing; keys look like {list(raw)[:3]}"

model = smp.FPN(encoder_name="resnet34", encoder_weights=None,
                in_channels=3, classes=len(FLAIR_CLASSES))
missing, unexpected = model.load_state_dict(state, strict=True)
print(f"loaded {len(state)} tensors into FPN/resnet34 with {len(FLAIR_CLASSES)} outputs")
if "criterion.weight" in raw:
    print("train class weights:", raw["criterion.weight"].numpy().astype(int).tolist())
model.eval()

### Load check — the README hands us a free one

*"classes n° 15, 16, 17 and 19 should appear at 0 in the logits and should not be
present in the final argmax product."* So if a deactivated class ever wins the
argmax, the weights did **not** load correctly. That turns a silent failure into
a loud one, which is the whole lesson of `bug-018`.

In [ ]:
with torch.no_grad():
    probe = model(torch.randn(2, 3, 512, 512))
print("logits:", tuple(probe.shape), "- expected (2, 19, 512, 512)")
assert probe.shape[1] == 19, "head is not 19-way; class mapping below would be wrong"

won = set(np.unique(probe.argmax(1).numpy()).tolist())
alive = won - set(DEAD_IDX)
print("classes winning argmax on noise:", sorted(won))
assert not (won & set(DEAD_IDX)), (
    f"deactivated classes {sorted(won & set(DEAD_IDX))} won the argmax — "
    "weights did not load, do not trust anything below")
print("load check passed —", len(alive), "active classes present")

## Run — tiled inference at both GSDs

The README is explicit: *"No data augmentation method concerning scale change was
used during training."* The model is scale-brittle by construction, so the
`upsampled` arm (matching its 0.2 m training scale) is the one with the stronger
prior — even though it fabricates detail. That is exactly why both arms run.

In [ ]:
@torch.no_grad()
def predict(img, mpp, mode):
    """Return a (H, W) class-index map at the ORIGINAL pixel grid.

    mode="native"    feed at the tile's own 0.386 m/px — objects look half size
    mode="upsampled" resample to FLAIR's 0.2 m/px — fabricates detail
    """
    arr = np.asarray(img, np.float32)
    h0, w0 = arr.shape[:2]
    if mode == "upsampled":
        scale = mpp / flair_mpp
        img_s = img.resize((int(w0 * scale), int(h0 * scale)), Image.BICUBIC)
        arr = np.asarray(img_s, np.float32)
    h, w = arr.shape[:2]

    x = (arr - MEAN) / STD
    logits = np.zeros((len(FLAIR_CLASSES), h, w), np.float32)
    count = np.zeros((h, w), np.float32)
    step = patch - overlap
    for top in range(0, max(h - overlap, 1), step):
        for left in range(0, max(w - overlap, 1), step):
            t, l = min(top, max(h - patch, 0)), min(left, max(w - patch, 0))
            tile = x[t:t + patch, l:l + patch]
            ph, pw = tile.shape[:2]
            if ph < patch or pw < patch:
                tile = np.pad(tile, ((0, patch - ph), (0, patch - pw), (0, 0)))
            out = model(torch.from_numpy(tile.transpose(2, 0, 1))[None])[0].numpy()
            logits[:, t:t + ph, l:l + pw] += out[:, :ph, :pw]
            count[t:t + ph, l:l + pw] += 1
    logits /= np.maximum(count, 1)
    pred = logits.argmax(0).astype(np.uint8)
    if mode == "upsampled":
        pred = np.array(Image.fromarray(pred).resize((w0, h0), Image.NEAREST))
    return pred


def fold(pred, ours):
    idx = [FLAIR_CLASSES.index(c) for c in TO_OURS[ours]]
    return np.isin(pred, idx)

In [ ]:
names = locations()[:n]
results, cache = [], {}

for name in names:
    img, grid, meta = load_location(name)
    for mode in gsd_modes:
        t0 = time.time()
        pred = predict(img, meta["mpp"], mode)
        cache[(name, mode)] = pred
        row = {"location": name, "mode": mode, "seconds": round(time.time() - t0, 1)}
        for ours, osm_cls in (("building", "building"), ("water", "water"), ("woody", "wood")):
            p, t = fold(pred, ours), osm_mask(name, osm_cls)
            row[ours] = {
                "pred_frac": float(p.mean()),
                "osm_frac": float(t.mean()),
                "iou": iou(p, t),
                "null_iou": null_iou(p, t),
                "recall": float((p & t).sum() / t.sum()) if t.sum() else float("nan"),
            }
        row["track"] = {"pred_frac": float(fold(pred, "track").mean())}
        results.append(row)
        b = row["building"]
        print(f"{name:30s} {mode:>10s} {row['seconds']:>5.0f}s  "
              f"building IoU {b['iou']:.3f} (null {b['null_iou']:.3f})  "
              f"claims {b['pred_frac']:.1%} vs actual {b['osm_frac']:.1%}")

(HERE / "results.json").write_text(json.dumps(results, indent=1))

## Cross-date consistency — scored with no ground truth at all

Every date of a location is co-registered to **1.5 cm median** and shares one
label. So running the model on two dates of the *same ground* gives a score that
needs no truth: **disagreement between dates is pure source error.**

This measures the property the dataset exists for — invariance to season, haze
and sensor date. A source that scores well against OSM but flips between dates is
keying on appearance, and is useless here however good its IoU looks.

In [ ]:
consistency = []
if cross_date:
    for name in names[:min(5, len(names))]:
        dates = sorted(p.stem for p in (ROOT / name).glob("*.jpg") if p.stem != "current")
        if not dates:
            continue
        other = dates[0]
        img_o, _, meta = load_location(name, other)
        for mode in gsd_modes:
            a = cache[(name, mode)]
            b = predict(img_o, meta["mpp"], mode)
            row = {"location": name, "mode": mode, "date": other,
                   "pixel_agreement": float((a == b).mean())}
            for ours in ("building", "water", "woody"):
                row[ours] = iou(fold(a, ours), fold(b, ours))
            consistency.append(row)
            print(f"{name:30s} {mode:>10s} vs {other}  pixels {row['pixel_agreement']:.1%}  "
                  f"building {row['building']:.3f}  woody {row['woody']:.3f}")

(HERE / "results-consistency.json").write_text(json.dumps(consistency, indent=1))

## Numbers

Distribution per class per GSD, each paired with its null (`bug-016`). The
`claims vs actual` column is the one that caught YOLOE: a mask covering half the
tile scores well on recall while carrying no positional information.

In [ ]:
import statistics as st

print(f"{'mode':>10s} {'class':>9s} {'n':>3s} {'IoU':>7s} {'null':>7s} {'recall':>7s} "
      f"{'claims':>7s} {'actual':>7s}")
for mode in gsd_modes:
    for cls in ("building", "water", "woody"):
        rows = [r[cls] for r in results if r["mode"] == mode
                and not np.isnan(r[cls]["iou"])]
        if not rows:
            continue
        m = lambda k: st.median([r[k] for r in rows if not np.isnan(r[k])])   # noqa: E731
        print(f"{mode:>10s} {cls:>9s} {len(rows):>3} {m('iou'):>7.3f} {m('null_iou'):>7.3f} "
              f"{m('recall'):>7.3f} {m('pred_frac'):>7.1%} {m('osm_frac'):>7.1%}")

print("\nbeats null (IoU > 2x null), per mode/class:")
for mode in gsd_modes:
    for cls in ("building", "water", "woody"):
        rows = [r[cls] for r in results if r["mode"] == mode]
        k = sum(1 for r in rows if r["iou"] > 2 * r["null_iou"])
        print(f"  {mode:>10s} {cls:>9s}  {k}/{len(rows)}")

## Look at it

Full resolution, every class, both GSDs, **before** the verdict. The specific
thing to check: do the masks follow buildings and treelines, or the 512 px
inference grid? Grid-aligned rectangles is what YOLOE produced and it is visible
instantly.

In [ ]:
palette = {"building": (1, 0, 0), "water": (0, 0.4, 1),
           "woody": (0, 0.8, 0), "track": (1, 0.85, 0)}
show = names[:min(3, len(names))]
cols = 1 + len(gsd_modes) + 1
fig, axes = plt.subplots(len(show), cols, figsize=(5 * cols, 5 * len(show)))
axes = np.atleast_2d(axes)
for row, name in enumerate(show):
    img, _, _ = load_location(name)
    arr = np.array(img)
    axes[row, 0].imshow(arr)
    axes[row, 0].set_title(f"{name}\nimagery", fontsize=8)
    for j, mode in enumerate(gsd_modes, start=1):
        over = np.zeros((*arr.shape[:2], 3))
        for cls, colour in palette.items():
            m = fold(cache[(name, mode)], cls)
            for c in range(3):
                over[..., c] = np.where(m, colour[c], over[..., c])
        axes[row, j].imshow(arr)
        axes[row, j].imshow(over, alpha=0.5)
        axes[row, j].set_title(f"FLAIR — {mode}", fontsize=8)
    ctrl = np.zeros((*arr.shape[:2], 3))
    ctrl[..., 0] = osm_mask(name, "building")
    ctrl[..., 2] = osm_mask(name, "water")
    axes[row, -1].imshow(arr)
    axes[row, -1].imshow(ctrl, alpha=0.6)
    axes[row, -1].set_title("OSM control (building/water)", fontsize=8)
    for a in axes[row]:
        a.axis("off")
plt.tight_layout()
plt.savefig(FIGS / "flair-overview.png", dpi=130, bbox_inches="tight")
plt.show()

## Verdict

`worked` / `failed` / `parked`, two or three lines, then copy into `notes.md`.

State the GSD answer separately from the transfer answer — they are different
findings and the next probe needs both.

### What is *not* established